In [1]:
%%capture
# Instalación de Unsloth optimizado para GPU T4 y librerías de fine-tuning
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets

In [2]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("[OK] Autenticado exitosamente con Hugging Face vía Kaggle Secrets.")
except Exception as e:
    print("[INFO] Ingresa tu token manualmente si te lo solicita:")
    login()

[OK] Autenticado exitosamente con Hugging Face vía Kaggle Secrets.


In [3]:
import torch
from unsloth import FastModel

max_seq_length = 2048

print("Descargando y cargando Gemma 4 31B en 4 bits nativos (QLoRA real)...")

# 1. Carga del modelo base 4-bit
model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-31B-it",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

# 2. Inyección de adaptadores LoRA (r=8, alpha=16, todas las proyecciones)
print("\nInyectando adaptador LoRA (r=8, alpha=16)...")
model = FastModel.get_peft_model(
    model,
    r=8,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # Atención
        "gate_proj", "up_proj", "down_proj"       # MLP SwiGLU
    ],
    lora_alpha=16,
    lora_dropout=0,                      # Optimizado a 0 en Unsloth para máxima velocidad
    bias="none",
    use_gradient_checkpointing="unsloth", # Ahorro masivo de VRAM
    random_state=42,
)

print("\n==========================================")
print("     ¡MODELO Y LORA LISTOS CON UNSLOTH!   ")
print("==========================================")
model.print_trainable_parameters()

# Verificar estado de VRAM
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / (1024**3)
    print(f"GPU {i}: {allocated:.2f} GB asignados")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Descargando y cargando Gemma 4 31B en 4 bits nativos (QLoRA real)...
==((====))==  Unsloth 2026.10.4: Fast Gemma4 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.
total weights      : 17.338 GiB
no_split classes   : ['Gemma4AudioLayer', 'Gemma4TextDecoderLayer', 'Gemma4VisionEncoderLayer', 'Gemma4VisionPatchEmbedder']
output head        : lm_head -> cuda:1
head headroom      : 0.500 GiB
activation reserve : 4.047 GiB requested
tie

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]


Inyectando adaptador LoRA (r=8, alpha=16)...

     ¡MODELO Y LORA LISTOS CON UNSLOTH!   
trainable params: 66,740,864 || all params: 31,339,827,376 || trainable%: 0.2130
GPU 0: 9.18 GB asignados
GPU 1: 8.86 GB asignados


In [4]:
import os
import torch
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling

# 1. Encontrar automáticamente la ruta de los archivos .jsonl
train_path = None
val_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f == "sft_train.jsonl":
            train_path = os.path.join(root, f)
        elif f == "sft_val.jsonl":
            val_path = os.path.join(root, f)

print(f"Ruta Train: {train_path}")
print(f"Ruta Val:   {val_path}")

dataset = load_dataset("json", data_files={"train": train_path, "val": val_path})
train_data = dataset["train"].select_columns(["text"])
val_data = dataset["val"].select_columns(["text"])

print(f"[OK] Datos cargados: {len(train_data)} train | {len(val_data)} val")

# 2. Extraer el tokenizer de texto real del Processor multimodal
text_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer

# 3. Collator Nativo con Loss Masking para Gemma
class GemmaCompletionCollator(DataCollatorForLanguageModeling):
    def __init__(self, tokenizer, response_template="<start_of_turn>model\n"):
        super().__init__(tokenizer=tokenizer, mlm=False)
        self.response_token_ids = tokenizer.encode(response_template, add_special_tokens=False)

    def torch_call(self, examples):
        batch = super().torch_call(examples)
        labels = batch["labels"]
        r_len = len(self.response_token_ids)
        for i in range(labels.shape[0]):
            input_ids = batch["input_ids"][i].tolist()
            idx = -1
            for j in range(len(input_ids) - r_len + 1):
                if input_ids[j : j + r_len] == self.response_token_ids:
                    idx = j + r_len
                    break
            if idx != -1:
                labels[i, :idx] = -100
        batch["labels"] = labels
        return batch

collator = GemmaCompletionCollator(tokenizer=text_tokenizer)
print("[OK] Collator de Loss Masking inicializado con éxito.")

Ruta Train: /kaggle/input/datasets/adrimonfer/gemma-sft-dataset/sft_train.jsonl
Ruta Val:   /kaggle/input/datasets/adrimonfer/gemma-sft-dataset/sft_val.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

[OK] Datos cargados: 741 train | 83 val
[OK] Collator de Loss Masking inicializado con éxito.


In [5]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./checkpoints_main_lora",
    dataset_text_field="text",
    max_length=1024,                    # 2x más rápido y 75% menos VRAM en atención
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,      # Batch efectivo = 8 muestras por paso
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    max_steps=60,                       # 60 pasos (~480 muestras): punto dulce anti-overfitting (~1 hora)
    fp16=True,
    logging_steps=5,                    # Veremos el Training Loss cada 5 pasos
    eval_strategy="no",                 # Desactivado para evitar el pico OOM de batch=8
    save_strategy="steps",
    save_steps=20,                      # Guarda checkpoints seguros en el paso 20, 40 y 60
    save_total_limit=3,
    weight_decay=0.01,
    packing=False,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    data_collator=collator,
    args=training_args
)

print("\n==========================================")
print("  ¡INICIANDO ENTRENAMIENTO LORA (60 STEPS)! ")
print("==========================================")

trainer_stats = trainer.train()

Unsloth: not enough free memory for dataset tokenization workers (~1GB each); tokenizing in-process.
Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"]:   0%|          | 0/741 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.



  ¡INICIANDO ENTRENAMIENTO LORA (60 STEPS)! 


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 741 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 66,740,864 of 31,339,827,376 (0.21% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,2.834698
10,1.523862
15,0.802885
20,0.612502
25,0.448792
30,0.535052
35,0.617349
40,0.499514
45,0.433937
50,0.406348


In [6]:
import os
import json

output_adapter_dir = "./main_lora"

# 1. Guardar los pesos LoRA (.safetensors) y configuración PEFT
print("Exportando adaptador LoRA en formato oficial PEFT (.safetensors)...")
model.save_pretrained(output_adapter_dir)

# 2. Asegurar que base_model_name_or_path coincida con el modelo oficial del arnés de Kaggle
config_path = os.path.join(output_adapter_dir, "adapter_config.json")
with open(config_path, "r", encoding="utf-8") as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = "google/gemma-4-31b-it-qat-w4a16-ct"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

# 3. Comprimir en un ZIP listo para descargar a tu PC
!zip -r main_lora_adapter.zip ./main_lora

print("\n========================================================")
print("  ¡ADAPTADOR EXPORTADO Y EMPAQUETADO CON ÉXITO!         ")
print("========================================================")
for f in os.listdir(output_adapter_dir):
    size_mb = os.path.getsize(os.path.join(output_adapter_dir, f)) / (1024**2)
    print(f" - {f}: {size_mb:.2f} MB")

Exportando adaptador LoRA en formato oficial PEFT (.safetensors)...
  adding: main_lora/ (stored 0%)
  adding: main_lora/adapter_config.json (deflated 71%)
  adding: main_lora/adapter_model.safetensors (deflated 13%)
  adding: main_lora/README.md (deflated 65%)

  ¡ADAPTADOR EXPORTADO Y EMPAQUETADO CON ÉXITO!         
 - adapter_config.json: 0.00 MB
 - adapter_model.safetensors: 254.77 MB
 - README.md: 0.01 MB
